# 02 Normalizar contexto, CPI y dos indicadores de Eurostat

Crear una sola tabla larga con los tres indicadores de contexto, con una fila por país, año e indicador.



In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
try:
    display
except NameError:
    display = print

RAIZ = Path.cwd().parent if Path.cwd().name == "01_notebooks" else Path.cwd()
FUENTES = RAIZ / "00_fuentes"
SILVER = RAIZ / "02_datos" / "silver"
SILVER.mkdir(parents=True, exist_ok=True)

def unico_excel(carpeta):
    archivos = list((FUENTES / carpeta).glob("*.xlsx"))
    assert len(archivos) == 1, f"Se esperaba un Excel en {carpeta} y hay {len(archivos)}"
    return archivos[0]

ARCHIVO_CPI = unico_excel("transparency_international_cpi_2025")
ARCHIVO_GASTO = unico_excel("eurostat_gasto_proteccion_social")
ARCHIVO_AROPE = unico_excel("eurostat_riesgo_pobreza_exclusion")

pd.set_option("display.width", 200)
registro = []
def registrar(paso, que_se_hizo, filas):
    registro.append({"paso": paso, "que_se_hizo": que_se_hizo, "filas": filas})

## CPI
Puntuación de 0 (muy corrupto) a 100 (muy limpio), con una columna por año. Se pasa a formato largo. Kosovo viene como KSV y en UNODC es XXK, así que se unifica.



In [ ]:
cpi = pd.read_excel(ARCHIVO_CPI, sheet_name="CPI Timeseries 2012 - 2025", header=3)
cpi = cpi.rename(columns={"Country / Territory": "pais", "ISO3": "iso3", "Region": "region_cpi"})
cpi["iso3"] = cpi["iso3"].str.strip().replace({"KSV": "XXK"})

columnas = [c for c in cpi.columns if re.fullmatch(r"CPI score \d{4}", str(c), flags=re.I)]
cpi_largo = cpi.melt(id_vars="iso3", value_vars=columnas, var_name="columna", value_name="valor")
cpi_largo["anio"] = cpi_largo["columna"].str.extract(r"(\d{4})").astype(int)
cpi_largo["indicador"] = "CPI"
cpi_largo = cpi_largo.dropna(subset=["valor"])[["iso3", "anio", "indicador", "valor"]]

print(f"{len(cpi_largo):,} filas, {cpi_largo.iso3.nunique()} países, años {cpi_largo.anio.min()} a {cpi_largo.anio.max()}")
registrar(1, "CPI en formato largo", len(cpi_largo))

paises = cpi[["iso3", "pais", "region_cpi"]].drop_duplicates("iso3")
NOMBRE_A_ISO3 = dict(zip(paises["pais"], paises["iso3"]))
NOMBRE_A_ISO3.update({"Kosovo*": "XXK", "Liechtenstein": "LIE"})

## Eurostat
Eurostat da los países por nombre, así que se pasan a iso3 con los nombres del CPI. Las filas de agregados (UE-27, zona euro) no son países y se dejan fuera.

In [ ]:
def leer_eurostat(archivo, fila_anios, indicador):
    r = pd.read_excel(archivo, sheet_name="Sheet 1", header=None)
    anios = {i: int(v) for i, v in r.iloc[fila_anios, 1:].items() if str(v).strip().isdigit()}
    filas, agregados = [], set()
    for k in range(fila_anios + 2, len(r)):
        nombre = r.iloc[k, 0]
        if pd.isna(nombre) or str(nombre).startswith(("Special", "Data", "Note", "Observation")):
            break
        if nombre not in NOMBRE_A_ISO3:
            agregados.add(nombre)
            continue
        for i, a in anios.items():
            v = pd.to_numeric(r.iloc[k, i], errors="coerce")
            if pd.notna(v):
                filas.append((NOMBRE_A_ISO3[nombre], a, indicador, float(v)))
    print(f"{indicador}: {len(agregados)} filas de agregados fuera")
    return pd.DataFrame(filas, columns=["iso3", "anio", "indicador", "valor"])

gasto = leer_eurostat(ARCHIVO_GASTO, 8, "GASTO_PS")
arope = leer_eurostat(ARCHIVO_AROPE, 9, "AROPE")
for d in (gasto, arope):
    print(f"{d.indicador.iloc[0]}: {len(d):,} filas, {d.iso3.nunique()} países, años {d.anio.min()} a {d.anio.max()}")
registrar(2, "Gasto en protección social (% PIB) en formato largo", len(gasto))
registrar(3, "Población en riesgo de pobreza o exclusión (%) en formato largo", len(arope))

## Unión y comprobaciones

In [ ]:
contexto = pd.concat([cpi_largo, gasto, arope], ignore_index=True)
contexto = contexto.sort_values(["indicador", "iso3", "anio"]).reset_index(drop=True)
assert not contexto.duplicated(["iso3", "anio", "indicador"]).any()
assert contexto["valor"].notna().all()
assert contexto["valor"].between(0, 100).all()
registrar(4, "Unión de los tres indicadores", len(contexto))

resumen = contexto.groupby("indicador").agg(filas=("valor", "size"), paises=("iso3", "nunique"),
                                            desde=("anio", "min"), hasta=("anio", "max"),
                                            minimo=("valor", "min"), maximo=("valor", "max"))
display(resumen)

Comprobación con una cifra conocida: el CPI 2025 de España debe coincidir con el publicado por Transparency International.

In [ ]:
display(contexto.query("iso3 == 'ESP' and anio == 2025"))

## Dimensión de indicadores
Una fila por indicador, con su nombre, unidad, escala y fuente. Se usa en el modelo para describir cada indicador.

In [ ]:
dim_indicador = pd.DataFrame([
    {"indicador": "CPI", "nombre": "Índice de percepción de la corrupción", "unidad": "puntos", "escala": "0 (muy corrupto) a 100 (muy limpio)", "fuente": "Transparency International, CPI 2025"},
    {"indicador": "GASTO_PS", "nombre": "Gasto en protección social", "unidad": "% del PIB", "escala": "0 a 100", "fuente": "Eurostat, tps00098"},
    {"indicador": "AROPE", "nombre": "Población en riesgo de pobreza o exclusión social", "unidad": "% de la población", "escala": "0 a 100", "fuente": "Eurostat, ilc_peps01n"},
])
assert set(dim_indicador["indicador"]) == set(contexto["indicador"])
display(dim_indicador)

## Exportar


In [ ]:
contexto.to_csv(SILVER / "contexto_largo.csv", index=False, encoding="utf-8-sig", float_format="%.4f")
paises.to_csv(SILVER / "contexto_paises.csv", index=False, encoding="utf-8-sig")
dim_indicador.to_csv(SILVER / "dim_indicador.csv", index=False, encoding="utf-8-sig")
log = pd.DataFrame(registro)
log.to_csv(SILVER / "log_contexto.csv", index=False, encoding="utf-8-sig")
display(log)